In [147]:
push_forward_dir =  "/Users/anthonypoole/data/local_dqfno/test/"

In [200]:
import os
import glob
import random
from typing import Dict, Any, Tuple

import torch
from torch.utils.data import Dataset
import h5py

# Dummy functions for model and loss.
def model(x: torch.Tensor) -> int:
    return 5

def loss(x: Any, y: Any) -> int:
    return 2

class PushForwardDataSet(Dataset):
    """
    Dataset class for loading HDF5 files containing simulation data.
    Expects each file to have the datasets: 'density', 'omega', 'phi', and 'gamma_n'.
    
    Files are opened using context managers to guarantee that file handles are closed
    immediately after reading.
    """

    def __init__(self, directory: str):
        self.files = sorted(glob.glob(os.path.join(directory, '*.h5')))

    @property
    def files(self) -> list:
        return self._files

    @files.setter
    def files(self, file_list: list):
        self._files = file_list

    def __len__(self) -> int:
        return len(self.files)

    def _get_data_dict(self, f: h5py.File) -> Dict[str, Any]:
        """
        Returns a dictionary with the full data loaded from an open h5py.File.
        This loads the data into memory so that the file can be closed safely.
        """
        return {
            'density': f['density'][()],
            'omega': f['omega'][()],
            'phi': f['phi'][()],
            'gamma_n': f['gamma_n'][()]
        }

    def __getitem__(self, index: int) -> Dict[str, Any]:
        """
        Loads the HDF5 file at the specified index and returns a dictionary of numpy arrays.
        The file is opened and closed automatically.
        """
        file_path = self.files[index]
        with h5py.File(file_path, 'r') as f:
            data_dict = self._get_data_dict(f)
        return data_dict

    def get_chunk(self, file_path: str, pf_steps: int, chunk_size: int) -> torch.Tensor:
        """
        For the provided file, pf_steps, and chunk_size,
        returns a tensor of indices for chunked data extraction.

        This method opens the file only to calculate the number of timesteps,
        then closes it immediately.
        """
        with h5py.File(file_path, 'r') as f:
            data_timesteps = len(f['gamma_n'])
        total_timesteps = pf_steps * chunk_size
        diff = data_timesteps - total_timesteps
        # Randomly choose an end_index ensuring room for a complete chunk.
        end_index = random.randint(pf_steps * chunk_size, diff)
        start_index = end_index - pf_steps * chunk_size
        return torch.linspace(start_index, end_index - chunk_size, pf_steps).int()

    def get_data(self, file_path: str, start_idx: int, chunk_size: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Loads a chunk of data from the file starting at start_idx with the given chunk_size.
        The file is automatically closed after reading.
        
        Returns:
            state: Tensor concatenating density, omega, and phi (shape: [3, chunk_size]).
            gamma_n: Tensor for gamma_n (shape: [1, chunk_size]).
        """
        with h5py.File(file_path, 'r') as f:
            density = torch.from_numpy(f['density'][start_idx:start_idx + chunk_size]).unsqueeze(0)
            omega = torch.from_numpy(f['omega'][start_idx:start_idx + chunk_size]).unsqueeze(0)
            phi = torch.from_numpy(f['phi'][start_idx:start_idx + chunk_size]).unsqueeze(0)
            gamma_n = torch.from_numpy(f['gamma_n'][start_idx:start_idx + chunk_size]).unsqueeze(0) # (B, T)
        state = torch.cat((density, omega, phi)).unsqueeze(0).unsqueeze(0) # (B,C,T,V,X,Y)
        return state, gamma_n


In [201]:
pf_steps = 5
chunk_size = 100

# Create dataset instance
dataset = PushForwardDataSet(push_forward_dir)
for file in dataset.files:
    # Get starting indices for the first file.
    start_idxes = dataset.get_chunk(file, pf_steps, chunk_size)
    # Get the initial data chunk.
    input_data = dataset.get_data(file, start_idxes[0].item(), chunk_size)
    print(input_data[0].shape, input_data[1].shape)
    # Iterate over additional chunks and pass through the model.
    for start_idx in dataset.get_chunk(file, pf_steps, chunk_size):
        input_data = model(input_data)

    target_data = dataset.get_data(file, start_idxes[-1].item(), chunk_size)
    loss_value = loss(input_data, target_data)
    print(f"Loss: {loss_value}")

torch.Size([1, 1, 3, 100, 64, 64]) torch.Size([1, 100])
Loss: 2
torch.Size([1, 1, 3, 100, 64, 64]) torch.Size([1, 100])
Loss: 2


In [202]:
# =======================
# Unit tests start here.
# =======================
import unittest
import tempfile
import numpy as np
import shutil

class TestPushForwardDataSet(unittest.TestCase):
    def setUp(self):
        # Create a temporary directory with some dummy HDF5 files.
        self.test_dir = tempfile.mkdtemp()
        self.num_files = 3
        self.timesteps = 60  # For most tests, 60 timesteps are sufficient.
        for i in range(self.num_files):
            file_path = os.path.join(self.test_dir, f'test_{i}.h5')
            with h5py.File(file_path, 'w') as f:
                f.create_dataset('density', data=np.arange(self.timesteps))
                f.create_dataset('omega', data=np.arange(self.timesteps) + 10)
                f.create_dataset('phi', data=np.arange(self.timesteps) + 20)
                f.create_dataset('gamma_n', data=np.arange(self.timesteps) + 30)
        self.dataset = PushForwardDataSet(self.test_dir)

    def tearDown(self):
        # Remove temporary directory and files after tests.
        shutil.rmtree(self.test_dir)

    def test_length(self):
        # Verify that the dataset length matches the number of created files.
        self.assertEqual(len(self.dataset), self.num_files)

    def test_getitem(self):
        # Test __getitem__ to ensure it returns a dictionary with expected keys.
        data_dict = self.dataset[0]
        for key in ['density', 'omega', 'phi', 'gamma_n']:
            self.assertIn(key, data_dict)
        # Check that the 'density' dataset has the expected data.
        np.testing.assert_array_equal(data_dict['density'][:], np.arange(self.timesteps))

    def test_get_data(self):
        # Test get_data for correct tensor shapes and values.
        pf_steps = 5
        chunk_size = 10
        start_idx = 0
        state, gamma_n = self.dataset.get_data(self.dataset.files[0], start_idx, chunk_size)
        # state should be (3, chunk_size) and gamma_n should be (1, chunk_size).
        self.assertEqual(state.shape, (3, chunk_size))
        self.assertEqual(gamma_n.shape, (1, chunk_size))
        np.testing.assert_array_equal(state[0].numpy(), np.arange(start_idx, start_idx + chunk_size))
        np.testing.assert_array_equal(state[1].numpy(), np.arange(start_idx, start_idx + chunk_size) + 10)
        np.testing.assert_array_equal(state[2].numpy(), np.arange(start_idx, start_idx + chunk_size) + 20)
        np.testing.assert_array_equal(gamma_n[0].numpy(), np.arange(start_idx, start_idx + chunk_size) + 30)

    def test_get_chunk(self):
        # For get_chunk to work, total_timesteps must be less than available timesteps.
        # Create a file with more timesteps.
        pf_steps = 5
        chunk_size = 10
        timesteps_valid = 120  # Enough timesteps so that pf_steps * chunk_size < timesteps_valid.
        file_path = os.path.join(self.test_dir, 'test_valid.h5')
        with h5py.File(file_path, 'w') as f:
            f.create_dataset('density', data=np.arange(timesteps_valid))
            f.create_dataset('omega', data=np.arange(timesteps_valid) + 10)
            f.create_dataset('phi', data=np.arange(timesteps_valid) + 20)
            f.create_dataset('gamma_n', data=np.arange(timesteps_valid) + 30)
        # Test that get_chunk returns a tensor with pf_steps elements.
        indices = self.dataset.get_chunk(file_path, pf_steps, chunk_size)
        self.assertTrue(torch.is_tensor(indices))
        self.assertEqual(indices.numel(), pf_steps)

    def test_model_and_loss(self):
        # Test the dummy model and loss functions.
        x = torch.tensor(1)
        y = torch.tensor(2)
        self.assertEqual(model(x), 5)
        self.assertEqual(loss(x, y), 2)